In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir('/home/jovyan/work/ufc/')

In [3]:
import pandas as pd
from ruamel.yaml import YAML
import numpy as np
import click
import time

import os
import re

from functools import partial
from fastcore.basics import chunked

import sys
sys.path.append('.')

from src.stat_funcs import get_stat_feat, last_el
from src.constants import ActivityLogger

from pandarallel import pandarallel

pandarallel.initialize(progress_bar=False, nb_workers=os.cpu_count())

start = time.time()


conf = YAML().load(open('params.yaml'))

fights_df = pd.read_csv(conf['filter']['filt_fights_fn'])

INFO: Pandarallel will run on 1 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [12]:

fights_df2 = shrink_df(fights_df)

In [9]:
for data_type in fights_df.dtypes.astype(str).unique():
    if in

array(['object', 'int64', 'float64'], dtype=object)

In [ ]:
fights_df.sort_values(by=['Fighter', 'event_date'], inplace=True)
# stat_cols = [it for it in fights_df.columns if re.search('_stat', it)]
stat_cols = [it for it in fights_df.columns if '_stat' in it]

# stat_cols = [it for it in fights_df.columns if re.search('_stat', it) and all([not col in it for col in ['dob_stat', 'height_stat', 'reach_stat']])]

# aggs = ['sum', 'mean', 'max', 'min', 'std']
# cust_aggs = [('max_min_d', lambda x: x.max()-x.min())] + [(f'q_{l:.1f}', lambda x, l=l: x.quantile(l)) for l in np.arange(0.1, 1,0.5)]

aggs = eval(conf['stat_feat_gen']['aggs'])
cust_aggs = eval(conf['stat_feat_gen']['cust_aggs'])


fighters_chunks = list(chunked(fights_df['Fighter'].unique(), n_chunks=os.cpu_count()))
for i in range(len(fighters_chunks)):
    fights_df.loc[fights_df.Fighter.isin(fighters_chunks[i]), 'chunk'] = i


def get_stat_feat_chunk(df, fn, aggs, cust_aggs, stat_cols, n_shift, conf):
    
    return df.groupby('Fighter').apply(lambda x: fn(x, aggs = aggs, cust_aggs=cust_aggs, cols = stat_cols,
                                                min_per_num=conf['stat_feat_gen']['min_fights_num'],
                                                rol_window_size=conf['stat_feat_gen']['rol_window_size'], n_shift=n_shift))

visible_get_stat_feat_chunk = partial(get_stat_feat_chunk, fn=get_stat_feat, aggs=aggs, cust_aggs=cust_aggs, stat_cols=stat_cols, n_shift=1, conf=conf)